
# 練習問題: schedule(runtime) で素数カウントの割り当て方を比べる

## 目標

繰り返しごとに**仕事量が偏る**ループ (素数の判定は大きい数ほど時間がかかる) を題材に, `schedule(runtime)` を使えば**再コンパイルなしに** `OMP_SCHEDULE` 環境変数だけで割り当て方 (static / dynamic / guided) を切り替えて性能を比較できることを体験する.

## 課題

`count_primes.cpp` (または `count_primes.f90`) は, `2` から `N` までの整数のうち素数の個数を試し割り (trial division) で数える. 大きい数ほど判定に時間がかかるため, 繰り返しの仕事量は一様ではない.

コメント `TODO` の指示に従ってループを並列化せよ. `schedule` には `runtime` を指定する.

- C++: ループの直前に `#pragma omp parallel for schedule(runtime) reduction(+:count)` を加える.
- Fortran: `do` ループを `!$omp parallel do schedule(runtime) reduction(+:count)` と `!$omp end parallel do` で囲む.

(`reduction(+:count)` は素数の個数の総和を競合なく集計するためのもの.)

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore count_primes.cpp -o count_primes.exe

# Fortran
nvfortran -fast -mp=multicore count_primes.f90 -o count_primes.exe
```

`schedule(runtime)` なので, **コンパイルし直さず** `OMP_SCHEDULE` を変えるだけで割り当て方を切り替えられる. 実行時間を `time` で測って比べよ:

```
export OMP_NUM_THREADS=4

OMP_SCHEDULE=static          time ./count_primes.exe 300000
OMP_SCHEDULE=dynamic         time ./count_primes.exe 300000
OMP_SCHEDULE=dynamic,100     time ./count_primes.exe 300000
OMP_SCHEDULE=guided          time ./count_primes.exe 300000
```

## 期待される結果と考察

- 素数の個数 (`number of primes <= 300000 : 25997`) は, どの schedule でも**同じ**になることを確認せよ.
- 実行時間は schedule によって変わる. `static` は各スレッドに連番の塊を等分するため, 大きい数 (重い) を担当したスレッドだけ遅くなり, 全体が遅い側に引きずられる. `dynamic` や `guided` は終わったスレッドが次の塊を取りに行くので負荷の偏りが平準化され, 一般に速くなる.
- いくつかの schedule を試し, 最も速かったものを選べ. `dynamic,100` のようにチャンクサイズを変えると挙動が変わることも確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ count_primes.cpp
#include <cstdio>
#include <cstdlib>

int is_prime(long k) {
  if (k < 2) return 0;
  for (long d = 2; d * d <= k; d++) {
    if (k % d == 0) return 0;
  }
  return 1;
}

int main(int argc, char ** argv) {
  long N = (argc > 1 ? atol(argv[1]) : 300000L);
  long count = 0;
  // TODO: 下のループを #pragma omp parallel for schedule(runtime) reduction(+:count) で並列化せよ.
  for (long i = 2; i <= N; i++) {
    count += is_prime(i);
  }
  printf("number of primes <= %ld : %ld\n", N, count);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore count_primes.cpp -o count_primes_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./count_primes_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ count_primes.f90
module prime_mod
contains
  function is_prime(k) result(r)
    integer(8), intent(in) :: k
    integer :: r
    integer(8) :: d
    if (k < 2) then
       r = 0
       return
    end if
    d = 2
    do while (d * d <= k)
       if (mod(k, d) == 0) then
          r = 0
          return
       end if
       d = d + 1
    end do
    r = 1
  end function is_prime
end module prime_mod

program count_primes
  use prime_mod
  character(len=64) :: arg
  integer(8) :: N, i, count
  N = 300000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N
  end if
  count = 0
  ! TODO: 下のループを !$omp parallel do schedule(runtime) reduction(+:count) で並列化せよ.
  do i = 2, N
     count = count + is_prime(i)
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  print "(a,i0,a,i0)", "number of primes <= ", N, " : ", count
end program count_primes

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore count_primes.f90 -o count_primes_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./count_primes_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:count_primes.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:count_primes.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:count_primes.cpp}